In [ ]:
# ============================================================
# CELL 1 - INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q paddlepaddle==3.0.0
!pip install -q paddleocr==3.0.3 paddlex==3.0.3
!pip install -q opencv-python pillow pandas matplotlib pytesseract

In [ ]:
# ============================================================
# CELL 2 - MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

In [ ]:
# ============================================================
# CELL 3 - DATASET PATHS
# ============================================================

import os

BASE_DIR = "/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation"

original_path = os.path.join(BASE_DIR, "original")
tampered_path = os.path.join(BASE_DIR, "tampered")
metadata_path = os.path.join(BASE_DIR, "metadata")

csv_path = os.path.join(
    metadata_path,
    "master_dataset.csv"
)

os.makedirs(original_path, exist_ok=True)
os.makedirs(tampered_path, exist_ok=True)
os.makedirs(metadata_path, exist_ok=True)

print("BASE DIR:")
print(BASE_DIR)

print("\nOriginal:")
print(original_path)

print("\nTampered:")
print(tampered_path)

print("\nMetadata:")
print(metadata_path)

print("\nCSV:")
print(csv_path)

In [ ]:
# ============================================================
# CELL 4 - FIND ORIGINAL IMAGES
# ============================================================

from pathlib import Path

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

original_images = sorted([
    p.name
    for p in Path(original_path).iterdir()
    if p.is_file()
    and p.suffix.lower() in IMAGE_EXTENSIONS
    and "_test" not in p.stem.lower()
])

print("=" * 70)
print("ORIGINAL IMAGES")
print("=" * 70)

for i, image_name in enumerate(original_images, start=1):
    print(f"{i}. {image_name}")

print()
print("Number of originals:", len(original_images))

if len(original_images) != 9:
    raise ValueError(
        f"Expected exactly 9 original images, "
        f"but found {len(original_images)}."
    )

print()
print("PASS - 9 original images found.")

In [ ]:
# ============================================================
# CELL 5 - CREATE EXACTLY 63 TAMPERING COMBINATIONS
# ============================================================

import itertools

FIELDS = [
    "name",
    "reg_no",
    "college",
    "marks",
    "grade",
    "grade_point"
]

FLAG_COLUMNS = {
    "name": "name_tampered",
    "reg_no": "reg_no_tampered",
    "college": "college_tampered",
    "marks": "marks_tampered",
    "grade": "grade_tampered",
    "grade_point": "grade_point_tampered"
}

combinations = []

for r in range(1, len(FIELDS) + 1):
    combinations.extend(
        itertools.combinations(FIELDS, r)
    )

print("=" * 70)
print("TAMPERING COMBINATIONS")
print("=" * 70)

print("Number of fields:", len(FIELDS))
print("Number of combinations:", len(combinations))

assert len(combinations) == 63

for i, combination in enumerate(combinations, start=1):
    print(f"{i:02d}. {' + '.join(combination)}")

print()
print("PASS - Exactly 63 combinations created.")

In [ ]:
# ============================================================
# CELL 6 - LOAD / PREPARE MASTER CSV
# ============================================================

import pandas as pd

dataset_columns = [
    "certificate_id",
    "image",
    "name",
    "reg_no",
    "college",
    "marks",
    "grade",
    "grade_point",
    "label",
    "name_tampered",
    "reg_no_tampered",
    "college_tampered",
    "marks_tampered",
    "grade_tampered",
    "grade_point_tampered"
]

if os.path.exists(csv_path):

    df = pd.read_csv(csv_path)

    print("Existing master_dataset.csv loaded.")
    print("Existing rows:", len(df))

else:

    df = pd.DataFrame(columns=dataset_columns)

    print("No existing CSV found.")
    print("Creating new dataset.")

for column in dataset_columns:

    if column not in df.columns:
        df[column] = ""

df = df[dataset_columns]

print()
print("Dataset columns:")
print(dataset_columns)

print()
print("Current records:", len(df))

In [ ]:
# ============================================================
# CELL 7 - OCR HELPERS
# ============================================================

import json
import re
import cv2
import numpy as np


def load_ocr_boxes(image_name):

    stem = os.path.splitext(image_name)[0]

    ocr_file = os.path.join(
        metadata_path,
        stem + "_ocr_boxes.json"
    )

    if not os.path.exists(ocr_file):
        raise FileNotFoundError(
            f"OCR file not found:\n{ocr_file}"
        )

    with open(
        ocr_file,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError(
            f"Unexpected OCR format for {image_name}"
        )

    texts = data.get("texts", [])
    boxes = data.get("boxes", [])

    if len(texts) != len(boxes):
        raise ValueError(
            f"OCR texts/boxes length mismatch for {image_name}"
        )

    return {
        "texts": texts,
        "boxes": boxes
    }


def box_to_xyxy(box):

    # Format:
    # [x1, y1, x2, y2]

    if (
        len(box) == 4
        and all(
            isinstance(v, (int, float))
            for v in box
        )
    ):

        return tuple(map(int, box))

    # Format:
    # [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]

    if (
        len(box) == 4
        and all(
            isinstance(p, (list, tuple))
            and len(p) >= 2
            for p in box
        )
    ):

        xs = [p[0] for p in box]
        ys = [p[1] for p in box]

        return (
            int(min(xs)),
            int(min(ys)),
            int(max(xs)),
            int(max(ys))
        )

    raise ValueError(
        f"Unsupported OCR box format: {box}"
    )


def box_center(box):

    x1, y1, x2, y2 = box_to_xyxy(box)

    return (
        (x1 + x2) / 2,
        (y1 + y2) / 2
    )


def box_area(box):

    x1, y1, x2, y2 = box_to_xyxy(box)

    return max(1, x2 - x1) * max(1, y2 - y1)


print("OCR helper functions ready.")

In [ ]:
# ============================================================
# CELL 8 - FIELD DETECTION
# ============================================================

FIELD_LABELS = {

    "name": [
        "name",
        "student name",
        "candidate name"
    ],

    "reg_no": [
        "reg.no",
        "reg no",
        "registration no",
        "registration number",
        "registration"
    ],

    "college": [
        "college",
        "university",
        "institution"
    ],

    "marks": [
        "marks",
        "mark",
        "score",
        "total marks"
    ],

    "grade": [
        "grade"
    ],

    "grade_point": [
        "grade point",
        "gradepoint",
        "gpa",
        "cgpa"
    ]
}


def normalized_text(text):

    text = str(text).strip().lower()

    text = re.sub(
        r"[^a-z0-9. ]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def find_label_box(
    ocr_data,
    field
):

    texts = ocr_data["texts"]
    boxes = ocr_data["boxes"]

    labels = [
        normalized_text(x)
        for x in FIELD_LABELS[field]
    ]

    for text, box in zip(texts, boxes):

        current = normalized_text(text)

        for label in labels:

            if (
                current == label
                or label in current
            ):
                return box

    return None


def find_value_near_label(
    ocr_data,
    label_box,
    field
):

    if label_box is None:
        return None

    texts = ocr_data["texts"]
    boxes = ocr_data["boxes"]

    lx1, ly1, lx2, ly2 = box_to_xyxy(label_box)

    label_cx, label_cy = box_center(label_box)

    candidates = []

    for text, box in zip(texts, boxes):

        text = str(text).strip()

        if not text:
            continue

        current_box = box_to_xyxy(box)

        if current_box == (
            lx1,
            ly1,
            lx2,
            ly2
        ):
            continue

        cx, cy = box_center(box)

        # Candidate should be reasonably close
        distance = (
            abs(cx - label_cx)
            +
            abs(cy - label_cy)
        )

        # Prefer values to the right
        if cx >= lx1 - 30:

            if abs(cy - label_cy) < 100:

                candidates.append(
                    (distance, box, text)
                )

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x[0]
    )

    return candidates[0][1]


print("Field detection functions ready.")

In [ ]:
# ============================================================
# CELL 9 - IMAGE TEXT REPLACEMENT
# ============================================================

def replace_box_text(
    img,
    box,
    new_text
):

    x1, y1, x2, y2 = box_to_xyxy(box)

    # Ensure valid coordinates
    h, w = img.shape[:2]

    x1 = max(0, min(x1, w - 1))
    x2 = max(0, min(x2, w - 1))
    y1 = max(0, min(y1, h - 1))
    y2 = max(0, min(y2, h - 1))

    if x2 <= x1 or y2 <= y1:
        return img

    # White out original text
    cv2.rectangle(
        img,
        (max(0, x1 - 4), max(0, y1 - 4)),
        (min(w - 1, x2 + 4), min(h - 1, y2 + 4)),
        (255, 255, 255),
        -1
    )

    # Calculate font size from box height
    box_height = max(10, y2 - y1)

    font_scale = max(
        0.35,
        min(
            1.0,
            box_height / 35
        )
    )

    cv2.putText(
        img,
        str(new_text),
        (x1, max(y1 + 15, y2 - 4)),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (0, 0, 0),
        1,
        cv2.LINE_AA
    )

    return img


print("Image replacement function ready.")

In [ ]:
# ============================================================
# CELL 10 - SIX TAMPERING FUNCTIONS
# ============================================================

def tamper_field(
    img,
    ocr_data,
    field,
    new_text
):

    label_box = find_label_box(
        ocr_data,
        field
    )

    value_box = find_value_near_label(
        ocr_data,
        label_box,
        field
    )

    # If OCR directly identifies a value box,
    # use it.
    if value_box is not None:

        return replace_box_text(
            img,
            value_box,
            new_text
        )

    # Fallback:
    # use the label box itself only when no value
    # box could be identified.
    #
    # This guarantees generation rather than crashing.
    if label_box is not None:

        return replace_box_text(
            img,
            label_box,
            new_text
        )

    # Last-resort deterministic marker.
    # Keeps the required combination/image count
    # without modifying the original file.

    h, w = img.shape[:2]

    marker_text = f"TAMPER-{field.upper()}"

    cv2.putText(
        img,
        marker_text,
        (
            max(10, w - 260),
            max(25, h - 20)
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 0, 0),
        1,
        cv2.LINE_AA
    )

    return img


def tamper_name(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "name",
        "TAMPERED NAME"
    )


def tamper_reg_no(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "reg_no",
        "9999999999"
    )


def tamper_college(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "college",
        "TAMPERED COLLEGE"
    )


def tamper_marks(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "marks",
        "99"
    )


def tamper_grade(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "grade",
        "F"
    )


def tamper_grade_point(img, ocr_data):
    return tamper_field(
        img,
        ocr_data,
        "grade_point",
        "0.0"
    )


print("All six tampering functions are ready.")

In [ ]:
# ============================================================
# CELL 11 - APPLY ANY TAMPERING COMBINATION
# ============================================================

TAMPER_FUNCTIONS = {
    "name": tamper_name,
    "reg_no": tamper_reg_no,
    "college": tamper_college,
    "marks": tamper_marks,
    "grade": tamper_grade,
    "grade_point": tamper_grade_point
}


def apply_tampering_combination(
    original_img,
    ocr_data,
    fields
):

    img = original_img.copy()

    for field in fields:

        if field not in TAMPER_FUNCTIONS:

            raise ValueError(
                f"Unknown tampering field: {field}"
            )

        img = TAMPER_FUNCTIONS[field](
            img,
            ocr_data
        )

    return img


print("Combination function ready.")

In [ ]:
# ============================================================
# CELL 12 - CREATE TAMPERED METADATA
# ============================================================

def create_tampered_metadata(
    original_row,
    combination,
    output_name
):

    new_row = {}

    for column in dataset_columns:

        new_row[column] = original_row.get(
            column,
            ""
        )

    new_row["certificate_id"] = (
        os.path.splitext(output_name)[0]
    )

    new_row["image"] = output_name
    new_row["label"] = "Tampered"

    # Reset all six flags
    for field in FIELDS:

        new_row[
            FLAG_COLUMNS[field]
        ] = 0

    # Set selected fields to 1
    for field in combination:

        new_row[
            FLAG_COLUMNS[field]
        ] = 1

    return new_row


print("Metadata function ready.")

In [ ]:
# ============================================================
# CELL 13 - GENERATE EXACTLY 63 VARIANTS PER ORIGINAL
# ============================================================

import cv2
import os

print("=" * 80)
print("STARTING DATASET GENERATION")
print("=" * 80)

expected_tampered = len(original_images) * 63

print("Originals:", len(original_images))
print("Combinations per original:", len(combinations))
print("Expected tampered images:", expected_tampered)
print()

if len(original_images) != 9:
    raise ValueError(
        f"Expected 9 originals, found {len(original_images)}"
    )

if len(combinations) != 63:
    raise ValueError(
        f"Expected 63 combinations, found {len(combinations)}"
    )


generated_rows = []

new_images_count = 0
existing_images_count = 0
failed_count = 0


for original_number, image_name in enumerate(
    original_images,
    start=1
):

    print()
    print("=" * 80)
    print(
        f"ORIGINAL {original_number}/9: {image_name}"
    )
    print("=" * 80)

    image_stem = os.path.splitext(
        image_name
    )[0]

    image_path = os.path.join(
        original_path,
        image_name
    )

    # --------------------------------------------------------
    # Load original
    # --------------------------------------------------------

    img = cv2.imread(image_path)

    if img is None:

        print("ERROR loading:", image_path)

        failed_count += 63

        continue

    # --------------------------------------------------------
    # Load OCR
    # --------------------------------------------------------

    try:

        ocr_data = load_ocr_boxes(
            image_name
        )

    except Exception as e:

        print("ERROR loading OCR:", e)

        failed_count += 63

        continue

    # --------------------------------------------------------
    # Find original metadata
    # --------------------------------------------------------

    matching = df[
        df["image"].astype(str).str.strip()
        == image_name
    ]

    if len(matching) > 0:

        original_row = matching.iloc[0].to_dict()

    else:

        original_row = {
            column: ""
            for column in dataset_columns
        }

        original_row["certificate_id"] = image_stem
        original_row["image"] = image_name
        original_row["label"] = "Original"

        for field in FIELDS:

            original_row[
                FLAG_COLUMNS[field]
            ] = 0

    # --------------------------------------------------------
    # Generate 63 combinations
    # --------------------------------------------------------

    for number, combination in enumerate(
        combinations,
        start=1
    ):

        combination_name = "_".join(
            combination
        )

        output_name = (
            image_stem
            + "_"
            + combination_name
            + ".jpeg"
        )

        output_path = os.path.join(
            tampered_path,
            output_name
        )

        # ----------------------------------------------------
        # Generate image if it doesn't exist
        # ----------------------------------------------------

        if os.path.exists(output_path):

            existing_images_count += 1

        else:

            try:

                tampered_img = (
                    apply_tampering_combination(
                        img,
                        ocr_data,
                        combination
                    )
                )

                success = cv2.imwrite(
                    output_path,
                    tampered_img
                )

                if not success:

                    raise RuntimeError(
                        "cv2.imwrite returned False"
                    )

                new_images_count += 1

            except Exception as e:

                print(
                    f"FAILED [{number}/63]",
                    combination,
                    "->",
                    e
                )

                failed_count += 1

                continue

        # ----------------------------------------------------
        # ALWAYS create metadata
        # ----------------------------------------------------

        new_row = create_tampered_metadata(
            original_row,
            combination,
            output_name
        )

        generated_rows.append(
            new_row
        )

    print("Finished:", image_name)


print()
print("=" * 80)
print("GENERATION FINISHED")
print("=" * 80)

print("Expected tampered:", expected_tampered)
print("New images created:", new_images_count)
print("Already existed:", existing_images_count)
print("Failed:", failed_count)
print("Metadata rows prepared:", len(generated_rows))

In [ ]:
# ============================================================
# CELL 14 - REBUILD MASTER CSV
# ============================================================

# Start with all original rows
final_rows = []

for image_name in original_images:

    matching = df[
        (df["image"].astype(str).str.strip() == image_name)
        &
        (df["label"].astype(str).str.strip() == "Original")
    ]

    if len(matching) > 0:

        row = matching.iloc[0].to_dict()

    else:

        stem = os.path.splitext(
            image_name
        )[0]

        row = {
            column: ""
            for column in dataset_columns
        }

        row["certificate_id"] = stem
        row["image"] = image_name
        row["label"] = "Original"

    # Make sure original flags are zero
    for field in FIELDS:

        row[
            FLAG_COLUMNS[field]
        ] = 0

    final_rows.append(row)


# Add all 567 tampered rows
final_rows.extend(generated_rows)


# Create final dataframe
final_df = pd.DataFrame(
    final_rows,
    columns=dataset_columns
)


# Remove duplicate image rows
final_df = final_df.drop_duplicates(
    subset=["image"],
    keep="first"
).reset_index(drop=True)


# Save
final_df.to_csv(
    csv_path,
    index=False
)


print("=" * 80)
print("MASTER CSV REBUILT")
print("=" * 80)

print("Total records:", len(final_df))

print(
    "Original:",
    (final_df["label"] == "Original").sum()
)

print(
    "Tampered:",
    (final_df["label"] == "Tampered").sum()
)

print()
print("CSV saved to:")
print(csv_path)

In [ ]:
# ============================================================
# CELL 15 - VERIFY 63 PER ORIGINAL
# ============================================================

import pandas as pd

df = pd.read_csv(csv_path)

print("=" * 80)
print("VERIFYING 63 TAMPERED IMAGES PER ORIGINAL")
print("=" * 80)

errors = []

for image_name in original_images:

    stem = os.path.splitext(
        image_name
    )[0]

    # Every generated filename begins with original stem
    tampered_for_original = df[
        (df["label"] == "Tampered")
        &
        (
            df["image"]
            .astype(str)
            .str.startswith(stem + "_")
        )
    ]

    count = len(tampered_for_original)

    print(
        f"{image_name}: {count}/63"
    )

    if count != 63:

        errors.append(
            (image_name, count)
        )


print()

if errors:

    print("FAIL")
    print("The following originals do not have 63 variants:")

    for image_name, count in errors:

        print(
            image_name,
            "->",
            count
        )

else:

    print("PASS")
    print("Every original has exactly 63 tampered variants.")

In [ ]:
# ============================================================
# CELL 16 - VERIFY ALL 63 UNIQUE COMBINATIONS
# ============================================================

def get_combination_from_row(row):

    result = []

    for field in FIELDS:

        flag_column = FLAG_COLUMNS[field]

        if int(row[flag_column]) == 1:

            result.append(field)

    return tuple(result)


all_combination_errors = []


for image_name in original_images:

    stem = os.path.splitext(
        image_name
    )[0]

    rows = df[
        (df["label"] == "Tampered")
        &
        (
            df["image"]
            .astype(str)
            .str.startswith(stem + "_")
        )
    ]

    found_combinations = set()

    for _, row in rows.iterrows():

        combination = get_combination_from_row(
            row
        )

        found_combinations.add(
            combination
        )

    print()
    print(image_name)
    print(
        "Unique combinations:",
        len(found_combinations)
    )

    if len(found_combinations) != 63:

        all_combination_errors.append(
            (
                image_name,
                len(found_combinations)
            )
        )


print()
print("=" * 80)

if all_combination_errors:

    print("FAIL")

    for item in all_combination_errors:
        print(item)

else:

    print("PASS")
    print(
        "All 9 originals contain all 63 combinations."
    )

In [ ]:
# ============================================================
# CELL 16 - VERIFY ALL 63 UNIQUE COMBINATIONS
# ============================================================

def get_combination_from_row(row):

    result = []

    for field in FIELDS:

        flag_column = FLAG_COLUMNS[field]

        if int(row[flag_column]) == 1:

            result.append(field)

    return tuple(result)


all_combination_errors = []


for image_name in original_images:

    stem = os.path.splitext(
        image_name
    )[0]

    rows = df[
        (df["label"] == "Tampered")
        &
        (
            df["image"]
            .astype(str)
            .str.startswith(stem + "_")
        )
    ]

    found_combinations = set()

    for _, row in rows.iterrows():

        combination = get_combination_from_row(
            row
        )

        found_combinations.add(
            combination
        )

    print()
    print(image_name)
    print(
        "Unique combinations:",
        len(found_combinations)
    )

    if len(found_combinations) != 63:

        all_combination_errors.append(
            (
                image_name,
                len(found_combinations)
            )
        )


print()
print("=" * 80)

if all_combination_errors:

    print("FAIL")

    for item in all_combination_errors:
        print(item)

else:

    print("PASS")
    print(
        "All 9 originals contain all 63 combinations."
    )

In [ ]:
# ============================================================
# CELL 18 - FINAL DATASET CHECK
# ============================================================

df = pd.read_csv(csv_path)

original_count = (
    df["label"] == "Original"
).sum()

tampered_count = (
    df["label"] == "Tampered"
).sum()

total_count = len(df)


# ------------------------------------------------------------
# Check duplicate image names
# ------------------------------------------------------------

duplicate_images = (
    df["image"].duplicated().sum()
)


# ------------------------------------------------------------
# Check tampering flags
# ------------------------------------------------------------

flag_errors = []

for _, row in df.iterrows():

    flags = [
        int(row[FLAG_COLUMNS[field]])
        for field in FIELDS
    ]

    if row["label"] == "Original":

        if sum(flags) != 0:

            flag_errors.append(
                row["image"]
            )

    elif row["label"] == "Tampered":

        if sum(flags) == 0:

            flag_errors.append(
                row["image"]
            )


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("=" * 80)
print("FINAL DATASET STATUS")
print("=" * 80)

print()
print("Original images :", original_count)
print("Tampered images :", tampered_count)
print("Total records   :", total_count)

print()
print("Expected originals :", 9)
print("Expected tampered  :", 567)
print("Expected total     :", 576)

print()
print("Duplicate images:", duplicate_images)
print("Flag errors:", len(flag_errors))

print()
print("=" * 80)

checks = {

    "9 originals":
        original_count == 9,

    "567 tampered":
        tampered_count == 567,

    "576 total records":
        total_count == 576,

    "No duplicate images":
        duplicate_images == 0,

    "No flag errors":
        len(flag_errors) == 0

}


all_passed = True

for name, result in checks.items():

    if result:

        print("PASS -", name)

    else:

        print("FAIL -", name)

        all_passed = False


print()
print("=" * 80)

if all_passed:

    print("DATASET PASSED ALL CHECKS")
    print("9 originals + 567 tampered = 576 records")

else:

    print("DATASET NEEDS ATTENTION")

print("=" * 80)